# NB03-B — BGE-M3 Embedding & FAISS Index Building

**Purpose:** Embed both corpora with BGE-M3 (fp16) and build FAISS indexes. GPU-heavy; runs independently of NB03-A.

**Prerequisites:**
- NB03-A must have completed and its BM25 pickles uploaded to `crosslingual-rag-indexes`.
- Attach `crosslingual-rag-data` and `crosslingual-rag-indexes` as inputs.

**Outputs → `crosslingual-rag-indexes` Kaggle Dataset:**
```
indexes/faiss_bge_m3_hotpotqa/     (faiss.index + sparse.pkl + metadata.pkl)
indexes/faiss_bge_m3_wikipedia_id/ (faiss.index + sparse.pkl + metadata.pkl)
results/retrieval_sanity_check.json
```

**Est. GPU time:** ~2–3 hours on T4 (dominated by Wikipedia ID embedding ~20K docs).

**Checkpointing:** Wikipedia ID embedding saves every 2,000 docs to disk. If the session dies mid-way, re-run the notebook — already-embedded chunks are loaded from disk instead of re-computed.

**Why split from NB03-A?** BM25 build is CPU-only and finishes in ~30 min. Keeping it in the same session means the browser holds BM25 state in RAM while BGE-M3 loads ~2GB into VRAM, then FAISS accumulates another 1–2GB for 20K × 1024-dim float32 vectors. That combination reliably crashes the Kaggle Notebook editor around the 30-minute mark.

## 0. Install Dependencies

In [ ]:
!pip install -q FlagEmbedding faiss-cpu huggingface_hub rank_bm25 PySastrawi

## 1. Imports & GPU Check

In [ ]:
import os
import json
import pickle
import shutil
import time
import numpy as np
import faiss
from pathlib import Path
from tqdm import tqdm

from FlagEmbedding import BGEM3FlagModel
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠ No GPU detected — embedding will be very slow on CPU.")

## 2. Paths & Directories

In [ ]:
# ── Inputs ────────────────────────────────────────────────────────────────────
HOTPOTQA_DOCS_PATH = "/kaggle/input/crosslingual-rag-data/hotpotqa_source_docs.jsonl"
WIKIPEDIA_ID_PATH  = "/kaggle/input/crosslingual-rag-data/wikipedia_id_corpus.jsonl"
XQUAD_PARALLEL_PATH = "/kaggle/input/crosslingual-rag-data/xquad_id_parallel.jsonl"

# ── Outputs ───────────────────────────────────────────────────────────────────
WORKING_DIR     = Path("/kaggle/working")
INDEX_DIR       = WORKING_DIR / "indexes"
RESULTS_DIR     = WORKING_DIR / "results"
FAISS_HOTPOT    = INDEX_DIR / "faiss_bge_m3_hotpotqa"
FAISS_WIKI      = INDEX_DIR / "faiss_bge_m3_wikipedia_id"
CHECKPOINT_DIR  = WORKING_DIR / "wiki_embed_checkpoints"

for d in [INDEX_DIR, RESULTS_DIR, FAISS_HOTPOT, FAISS_WIKI, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Directories ready.")

# Verify inputs
for label, path in [("HotpotQA docs",  HOTPOTQA_DOCS_PATH),
                    ("Wikipedia ID",    WIKIPEDIA_ID_PATH),
                    ("XQuAD parallel",  XQUAD_PARALLEL_PATH)]:
    exists = os.path.exists(path)
    size   = os.path.getsize(path) / 1e6 if exists else 0
    print(f"  {label}: {'✓' if exists else '✗ MISSING'}  ({size:.1f} MB)")

## 3. Helpers

In [ ]:
def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def extract_text(record):
    """
    Extract a single text string from a document record.
    Handles: {text}, {content}, {title + text} → 'title. text'
    """
    title = record.get("title", "").strip()
    text  = record.get("text", record.get("content", "")).strip()
    if title and text:
        return f"{title}. {text}"
    return text or title


print("Helpers defined.")

## 4. Load Corpora

In [ ]:
print("Loading HotpotQA source docs...")
hotpotqa_docs  = load_jsonl(HOTPOTQA_DOCS_PATH)
hotpotqa_texts = [extract_text(d) for d in hotpotqa_docs]
hotpotqa_ids   = [d.get("id", str(i)) for i, d in enumerate(hotpotqa_docs)]
print(f"  {len(hotpotqa_texts)} docs | sample: {hotpotqa_texts[0][:100]}...")

empty = sum(1 for t in hotpotqa_texts if not t.strip())
if empty:
    print(f"  ⚠ {empty} empty texts")

print()
print("Loading Wikipedia ID corpus...")
wiki_docs  = load_jsonl(WIKIPEDIA_ID_PATH)
wiki_texts = [extract_text(d) for d in wiki_docs]
wiki_ids   = [d.get("id", str(i)) for i, d in enumerate(wiki_docs)]
print(f"  {len(wiki_texts)} docs | sample: {wiki_texts[0][:100]}...")

empty = sum(1 for t in wiki_texts if not t.strip())
if empty:
    print(f"  ⚠ {empty} empty texts")

## 5. Load BGE-M3 Model

In [ ]:
print("Loading BGEM3FlagModel (fp16)...")
bge_model = BGEM3FlagModel(
    'BAAI/bge-m3',
    use_fp16=True,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

device = next(bge_model.model.parameters()).device
print(f"Model loaded on: {device}")
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM used: {used:.2f} / {total:.1f} GB")

## 6. Embedding & FAISS Index Utilities

In [ ]:
def embed_batch(texts, batch_size=32, desc="Embedding"):
    """
    Embed texts with BGE-M3. Returns:
      - dense: np.ndarray (N, 1024) float32, L2-normalized
      - sparse: list of dicts {token_id: weight}
    """
    all_dense  = []
    all_sparse = []

    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch  = texts[i : i + batch_size]
        output = bge_model.encode(
            batch,
            batch_size=batch_size,
            max_length=512,
            return_dense=True,
            return_sparse=True,
            return_colbert_vecs=False
        )
        dense  = output['dense_vecs']      # (batch, 1024) float32
        sparse = output['lexical_weights'] # list of dicts

        # L2 normalize for cosine similarity via inner product
        norms = np.linalg.norm(dense, axis=1, keepdims=True)
        norms = np.where(norms == 0, 1.0, norms)
        dense = dense / norms

        all_dense.extend(dense.tolist())
        all_sparse.extend(sparse)

    return np.array(all_dense, dtype=np.float32), all_sparse


def build_faiss_index(dense_embeddings):
    """Inner-product FAISS index (exact search on L2-normalized vectors = cosine)."""
    dim   = dense_embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(dense_embeddings)
    print(f"  FAISS index: {index.ntotal} vectors, dim={dim}")
    return index


def save_bge_m3_index(faiss_index, sparse_weights, doc_ids, texts, out_dir):
    """
    Persist BGE-M3 index to out_dir/:
      faiss.index   — FAISS IndexFlatIP
      sparse.pkl    — list of sparse weight dicts
      metadata.pkl  — {doc_ids, texts}
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    faiss.write_index(faiss_index, str(out_dir / "faiss.index"))

    with open(out_dir / "sparse.pkl", "wb") as f:
        pickle.dump(sparse_weights, f)

    with open(out_dir / "metadata.pkl", "wb") as f:
        pickle.dump({"doc_ids": doc_ids, "texts": texts}, f)

    total_mb = sum(
        (out_dir / fn).stat().st_size / 1e6
        for fn in ["faiss.index", "sparse.pkl", "metadata.pkl"]
    )
    print(f"  ✓ Saved to {out_dir}/ (total {total_mb:.1f} MB)")


print("Utilities defined.")

## 7a. Embed & Index HotpotQA Source Docs

In [ ]:
print("=" * 55)
print("BGE-M3 Indexing: HotpotQA source docs")
print("=" * 55)

t0 = time.time()
hotpotqa_dense, hotpotqa_sparse = embed_batch(
    hotpotqa_texts, batch_size=32, desc="HotpotQA"
)
elapsed = (time.time() - t0) / 60
print(f"Embedding done in {elapsed:.1f} min")
print(f"Dense shape: {hotpotqa_dense.shape}")

faiss_hotpotqa = build_faiss_index(hotpotqa_dense)

save_bge_m3_index(
    faiss_hotpotqa, hotpotqa_sparse,
    hotpotqa_ids, hotpotqa_texts,
    FAISS_HOTPOT
)
print("✓ HotpotQA BGE-M3 index complete.")

# Free before Wikipedia embedding
del hotpotqa_dense, hotpotqa_sparse
torch.cuda.empty_cache()
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    print(f"VRAM after cleanup: {used:.2f} GB")

## 7b. Embed & Index Wikipedia ID Corpus (with checkpointing)

This is the heaviest cell: ~20K docs × 1024 dims ≈ ~2–3 hours on T4.

**Checkpointing:** every 2,000 docs are saved to `wiki_embed_checkpoints/chunk_NNNN.pkl`.
If the session dies mid-way, re-run this cell — already-finished chunks are loaded from disk.

In [ ]:
CHECKPOINT_SIZE = 2000  # docs per checkpoint batch


def embed_with_checkpoints(texts, batch_size=32, checkpoint_size=2000):
    """
    Embed corpus in chunks of checkpoint_size.
    Each chunk is persisted immediately. On re-run, existing chunks are loaded
    instead of re-embedded — enables resume without starting over.
    """
    all_dense  = []
    all_sparse = []
    n_chunks   = (len(texts) + checkpoint_size - 1) // checkpoint_size

    for chunk_idx in range(n_chunks):
        start      = chunk_idx * checkpoint_size
        end        = min(start + checkpoint_size, len(texts))
        ckpt_path  = CHECKPOINT_DIR / f"chunk_{chunk_idx:04d}.pkl"

        if ckpt_path.exists():
            print(f"  [Chunk {chunk_idx+1}/{n_chunks}] Resuming from checkpoint...")
            with open(ckpt_path, "rb") as f:
                ckpt = pickle.load(f)
            all_dense.extend(ckpt["dense"])
            all_sparse.extend(ckpt["sparse"])
            continue

        print(f"  [Chunk {chunk_idx+1}/{n_chunks}] Embedding docs {start}–{end}...")
        chunk_texts  = texts[start:end]
        chunk_dense  = []
        chunk_sparse = []

        for i in tqdm(range(0, len(chunk_texts), batch_size),
                      desc=f"Chunk {chunk_idx+1}", leave=False):
            batch  = chunk_texts[i : i + batch_size]
            output = bge_model.encode(
                batch,
                batch_size=batch_size,
                max_length=512,
                return_dense=True,
                return_sparse=True,
                return_colbert_vecs=False
            )
            dense  = output['dense_vecs']
            sparse = output['lexical_weights']

            norms = np.linalg.norm(dense, axis=1, keepdims=True)
            norms = np.where(norms == 0, 1.0, norms)
            dense = dense / norms

            chunk_dense.extend(dense.tolist())
            chunk_sparse.extend(sparse)

        with open(ckpt_path, "wb") as f:
            pickle.dump({"dense": chunk_dense, "sparse": chunk_sparse}, f)
        print(f"  ✓ Chunk {chunk_idx+1} checkpointed ({ckpt_path.name})")

        all_dense.extend(chunk_dense)
        all_sparse.extend(chunk_sparse)

    return np.array(all_dense, dtype=np.float32), all_sparse


print("=" * 55)
print("BGE-M3 Indexing: Wikipedia ID corpus")
print(f"  Total docs: {len(wiki_texts)}")
print(f"  Checkpoint every {CHECKPOINT_SIZE} docs")
print("=" * 55)

t0 = time.time()
wiki_dense, wiki_sparse = embed_with_checkpoints(
    wiki_texts, batch_size=32, checkpoint_size=CHECKPOINT_SIZE
)
elapsed = (time.time() - t0) / 60
print(f"\nTotal embedding time: {elapsed:.1f} min")
print(f"Dense shape: {wiki_dense.shape}")

In [ ]:
faiss_wiki = build_faiss_index(wiki_dense)

save_bge_m3_index(
    faiss_wiki, wiki_sparse,
    wiki_ids, wiki_texts,
    FAISS_WIKI
)
print("✓ Wikipedia ID BGE-M3 index complete.")

# Clean up checkpoints — they've served their purpose
shutil.rmtree(CHECKPOINT_DIR, ignore_errors=True)
print("Checkpoint directory removed.")

del wiki_dense, wiki_sparse
torch.cuda.empty_cache()
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    print(f"VRAM after cleanup: {used:.2f} GB")

## 8. RRF Fusion Utility

In [ ]:
# RRF is applied at query time in NB06/NB07 — defined here so the logic
# lives in one authoritative place.

def rrf_fuse(dense_ranking, sparse_ranking, k=60):
    """
    Reciprocal Rank Fusion of dense and sparse rankings.

    Args:
        dense_ranking:  list of (doc_idx, score) sorted desc by dense score
        sparse_ranking: list of (doc_idx, score) sorted desc by sparse score
        k:              RRF smoothing constant (default 60)
    Returns:
        list of (doc_idx, rrf_score) sorted desc by RRF score
    """
    scores = {}
    for rank, (doc_idx, _) in enumerate(dense_ranking, start=1):
        scores[doc_idx] = scores.get(doc_idx, 0.0) + 1.0 / (k + rank)
    for rank, (doc_idx, _) in enumerate(sparse_ranking, start=1):
        scores[doc_idx] = scores.get(doc_idx, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def sparse_score_query(query_sparse_weights, doc_sparse_weights):
    """
    BGE-M3 sparse similarity: sum of min(q_w, d_w) for shared tokens.
    Matches the BGE-M3 paper's lexical matching formulation.
    """
    score = 0.0
    for token_id, q_weight in query_sparse_weights.items():
        if token_id in doc_sparse_weights:
            score += min(q_weight, doc_sparse_weights[token_id])
    return score


# Sanity test
dense_ranks  = [(0, 0.9), (1, 0.8), (2, 0.6)]
sparse_ranks = [(1, 0.7), (0, 0.5), (3, 0.4)]
print("RRF test:", rrf_fuse(dense_ranks, sparse_ranks))
# Both doc 0 and doc 1 should rank above doc 2 or 3

## 9. Sanity Check: Cross-lingual Retrieval

In [ ]:
# 5 XQuAD parallel pairs → query Wikipedia ID corpus via BM25 and BGE-M3 Dense.
# Documents whether BM25 fails on EN→ID queries (expected: yes).
# This check is cited in the paper as empirical motivation for the architecture.

from rank_bm25 import BM25Okapi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stemmer_factory  = StemmerFactory()
stopword_factory = StopWordRemoverFactory()
stemmer   = stemmer_factory.create_stemmer()
stopwords = set(stopword_factory.get_stop_words())

def sastrawi_tokenize(text):
    text   = text.lower()
    tokens = text.split()
    tokens = [t for t in tokens if t not in stopwords]
    tokens = [stemmer.stem(t) for t in tokens]
    return tokens


print("Loading XQuAD parallel pairs...")
xquad_pairs = load_jsonl(XQUAD_PARALLEL_PATH)
print(f"  {len(xquad_pairs)} pairs")

import random
random.seed(42)
sample_pairs = random.sample(xquad_pairs, min(5, len(xquad_pairs)))
print(f"  Sampled {len(sample_pairs)} pairs for check")

In [ ]:
# Reload BM25 Wikipedia index for sanity check.
# We don't have the BM25 object in memory (built in NB03-A),
# so load from the crosslingual-rag-indexes dataset if attached,
# otherwise build inline from wiki_texts (already in memory).

BM25_WIKI_INPUT = "/kaggle/input/crosslingual-rag-indexes/bm25_wikipedia_id.pkl"

if os.path.exists(BM25_WIKI_INPUT):
    print("Loading BM25 Wikipedia index from crosslingual-rag-indexes dataset...")
    with open(BM25_WIKI_INPUT, "rb") as f:
        payload = pickle.load(f)
    bm25_wiki_check   = payload["bm25"]
    wiki_texts_check  = payload["texts"]
    print("  ✓ Loaded from dataset")
else:
    print("BM25 pkl not found in input — building inline from loaded wiki_texts...")
    tokenized = [sastrawi_tokenize(t) for t in tqdm(wiki_texts, desc="BM25 build")]
    bm25_wiki_check  = BM25Okapi(tokenized)
    wiki_texts_check = wiki_texts
    print("  ✓ Built inline")

# Reload FAISS wiki index (in case it was garbage-collected)
print("Reloading FAISS Wikipedia index...")
faiss_wiki_check = faiss.read_index(str(FAISS_WIKI / "faiss.index"))
print(f"  ✓ {faiss_wiki_check.ntotal} vectors")

In [ ]:
def retrieve_bm25(query, bm25_obj, texts, top_k=5):
    tokens = sastrawi_tokenize(query)
    if not tokens:
        return []
    import numpy as np
    scores   = bm25_obj.get_scores(tokens)
    top_idxs = np.argsort(scores)[::-1][:top_k]
    return [(int(idx), float(scores[idx]), texts[idx][:80]) for idx in top_idxs]


def retrieve_bge_dense(query, faiss_index, texts, top_k=5):
    import numpy as np
    output = bge_model.encode(
        [query],
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False
    )
    q_vec = output['dense_vecs'][0:1].astype(np.float32)
    norm  = np.linalg.norm(q_vec)
    if norm > 0:
        q_vec /= norm
    scores, idxs = faiss_index.search(q_vec, top_k)
    return [(int(idx), float(scores[0][i]), texts[idx][:80])
            for i, idx in enumerate(idxs[0]) if idx >= 0]


sanity_results = []
print("Cross-lingual sanity check:\n")

for i, pair in enumerate(sample_pairs):
    q_en   = pair.get('question_en', pair.get('question', ''))
    q_id   = pair.get('question_id_text', pair.get('question_id', ''))
    answer = pair.get('answer', 'N/A')

    bm25_en  = retrieve_bm25(q_en,  bm25_wiki_check, wiki_texts_check)
    bm25_id  = retrieve_bm25(q_id,  bm25_wiki_check, wiki_texts_check)
    dense_en = retrieve_bge_dense(q_en, faiss_wiki_check, wiki_texts_check)
    dense_id = retrieve_bge_dense(q_id, faiss_wiki_check, wiki_texts_check)

    result = {
        "pair_idx":            i,
        "question_en":         q_en,
        "question_id":         q_id,
        "answer":              answer,
        "bm25_en_top1_score":  bm25_en[0][1]  if bm25_en  else 0.0,
        "bm25_en_top1_text":   bm25_en[0][2]  if bm25_en  else "",
        "bm25_id_top1_score":  bm25_id[0][1]  if bm25_id  else 0.0,
        "bm25_id_top1_text":   bm25_id[0][2]  if bm25_id  else "",
        "dense_en_top1_score": dense_en[0][1] if dense_en  else 0.0,
        "dense_en_top1_text":  dense_en[0][2] if dense_en  else "",
        "dense_id_top1_score": dense_id[0][1] if dense_id  else 0.0,
        "dense_id_top1_text":  dense_id[0][2] if dense_id  else "",
    }
    sanity_results.append(result)

    print(f"Pair {i+1}: Q_EN = '{q_en[:70]}'")
    print(f"  BM25  EN→ID score: {result['bm25_en_top1_score']:.3f}  | {result['bm25_en_top1_text']}")
    print(f"  BM25  ID→ID score: {result['bm25_id_top1_score']:.3f}  | {result['bm25_id_top1_text']}")
    print(f"  Dense EN→ID score: {result['dense_en_top1_score']:.4f} | {result['dense_en_top1_text']}")
    print(f"  Dense ID→ID score: {result['dense_id_top1_score']:.4f} | {result['dense_id_top1_text']}")
    print()

In [ ]:
import numpy as np

avg_bm25_en  = np.mean([r['bm25_en_top1_score']  for r in sanity_results])
avg_bm25_id  = np.mean([r['bm25_id_top1_score']  for r in sanity_results])
avg_dense_en = np.mean([r['dense_en_top1_score'] for r in sanity_results])
avg_dense_id = np.mean([r['dense_id_top1_score'] for r in sanity_results])

summary = {
    "n_pairs":                  len(sanity_results),
    "avg_bm25_score_EN_query":  round(float(avg_bm25_en), 4),
    "avg_bm25_score_ID_query":  round(float(avg_bm25_id), 4),
    "bm25_gap_EN_minus_ID":     round(float(avg_bm25_en - avg_bm25_id), 4),
    "avg_dense_score_EN_query": round(float(avg_dense_en), 4),
    "avg_dense_score_ID_query": round(float(avg_dense_id), 4),
    "dense_gap_EN_minus_ID":    round(float(avg_dense_en - avg_dense_id), 4),
    "interpretation": (
        "Negative bm25_gap = BM25 scores lower on EN queries against ID corpus — confirms failure mode. "
        "dense_gap near 0 = BGE-M3 is language-agnostic as expected."
    ),
    "per_pair_results": sanity_results
}

print("=" * 50)
print("SANITY CHECK SUMMARY")
print("=" * 50)
print(f"  BM25  avg EN→ID: {avg_bm25_en:.4f}")
print(f"  BM25  avg ID→ID: {avg_bm25_id:.4f}")
print(f"  BM25  gap (EN−ID): {avg_bm25_en - avg_bm25_id:.4f}  ← expect negative")
print()
print(f"  Dense avg EN→ID: {avg_dense_en:.4f}")
print(f"  Dense avg ID→ID: {avg_dense_id:.4f}")
print(f"  Dense gap (EN−ID): {avg_dense_en - avg_dense_id:.4f}  ← expect ~0")

sanity_path = RESULTS_DIR / "retrieval_sanity_check.json"
with open(sanity_path, "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f"\nSaved to {sanity_path}")

## 10. Verify All Artifacts

In [ ]:
required_artifacts = [
    FAISS_HOTPOT / "faiss.index",
    FAISS_HOTPOT / "sparse.pkl",
    FAISS_HOTPOT / "metadata.pkl",
    FAISS_WIKI   / "faiss.index",
    FAISS_WIKI   / "sparse.pkl",
    FAISS_WIKI   / "metadata.pkl",
    RESULTS_DIR  / "retrieval_sanity_check.json",
]

all_ok = True
print("Artifact verification:")
for path in required_artifacts:
    exists = path.exists()
    size   = path.stat().st_size / 1e6 if exists else 0
    status = f"✓ ({size:.1f} MB)" if exists else "✗ MISSING"
    print(f"  {status}  {path.relative_to(WORKING_DIR)}")
    if not exists:
        all_ok = False

print()
if all_ok:
    print("All artifacts present. Ready to zip and persist.")
else:
    print("⚠ Missing artifacts — rerun the relevant cells above.")

## 11. Persist to Kaggle Dataset (`crosslingual-rag-indexes`)

In [ ]:
print("Zipping FAISS directories...")

for faiss_dir, zip_name in [
    (FAISS_HOTPOT, "faiss_bge_m3_hotpotqa"),
    (FAISS_WIKI,   "faiss_bge_m3_wikipedia_id"),
]:
    zip_path = WORKING_DIR / zip_name
    shutil.make_archive(
        str(zip_path), "zip",
        str(faiss_dir.parent),
        faiss_dir.name
    )
    size_mb = (WORKING_DIR / f"{zip_name}.zip").stat().st_size / 1e6
    print(f"  {zip_name}.zip ({size_mb:.1f} MB)")

shutil.copy(
    RESULTS_DIR / "retrieval_sanity_check.json",
    WORKING_DIR / "retrieval_sanity_check.json"
)
print("  retrieval_sanity_check.json")

print()
print("Files staged in /kaggle/working/:")
for f in sorted(WORKING_DIR.iterdir()):
    if f.is_file():
        print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")

print()
print("Next steps:")
print("  1. Notebook → Save & Run All  (commits outputs)")
print("  2. Upload both .zip files and retrieval_sanity_check.json")
print("     to the 'crosslingual-rag-indexes' Kaggle Dataset.")
print("  3. Proceed to NB04 — Reranker.")

## Quick-Load Snippet for Downstream Notebooks (NB04, NB06, NB07, NB08)

```python
import shutil, pickle, faiss
from pathlib import Path

IDX_INPUT = Path("/kaggle/input/crosslingual-rag-indexes")
FAISS_DIR = Path("/kaggle/working/faiss_db")
FAISS_DIR.mkdir(exist_ok=True)

# Unzip FAISS indexes
for zip_name, subdir in [
    ("faiss_bge_m3_hotpotqa.zip",     "faiss_bge_m3_hotpotqa"),
    ("faiss_bge_m3_wikipedia_id.zip", "faiss_bge_m3_wikipedia_id"),
]:
    shutil.unpack_archive(str(IDX_INPUT / zip_name), str(FAISS_DIR))

# Load FAISS dense indexes
faiss_hotpotqa = faiss.read_index(str(FAISS_DIR / "faiss_bge_m3_hotpotqa/faiss.index"))
faiss_wiki     = faiss.read_index(str(FAISS_DIR / "faiss_bge_m3_wikipedia_id/faiss.index"))

# Load sparse weights
with open(FAISS_DIR / "faiss_bge_m3_hotpotqa/sparse.pkl", "rb") as f:
    hotpotqa_sparse = pickle.load(f)
with open(FAISS_DIR / "faiss_bge_m3_wikipedia_id/sparse.pkl", "rb") as f:
    wiki_sparse = pickle.load(f)

# Load metadata
with open(FAISS_DIR / "faiss_bge_m3_hotpotqa/metadata.pkl", "rb") as f:
    hotpotqa_meta = pickle.load(f)  # {"doc_ids": [...], "texts": [...]}
with open(FAISS_DIR / "faiss_bge_m3_wikipedia_id/metadata.pkl", "rb") as f:
    wiki_meta = pickle.load(f)

# Load BM25 (from NB03-A)
with open(IDX_INPUT / "bm25_hotpotqa.pkl", "rb") as f:
    bm25_hotpotqa = pickle.load(f)["bm25"]
with open(IDX_INPUT / "bm25_wikipedia_id.pkl", "rb") as f:
    bm25_wiki = pickle.load(f)["bm25"]

print("All indexes loaded.")
```